# 🚁 Notebook 08 — PID Tuning

### How to actually *choose* Kp, Ki, Kd

You have a complete PID controller. The million-dollar question every engineer faces: **what
numbers do I put in?** Bad gains give you a sluggish drone, a wildly oscillating drone, or a
crash. Good gains give you fast, smooth, accurate flight.

Tuning has a reputation for being black magic. It isn't. This notebook gives you a **repeatable
workflow**, shows you each gain's effect **in isolation** via parameter sweeps, adds **automatic
performance metrics** so "good" becomes measurable, and demystifies the classic **Ziegler–Nichols**
recipe.

---

## 🎯 Learning objectives
- Follow a practical **manual tuning workflow** (P → D → I).
- See the isolated effect of `Kp`, `Ki`, `Kd` with **parameter sweeps**.
- Judge tuning quality with **automatic metrics** (rise time, overshoot, settling, steady-state error).
- Understand the **Ziegler–Nichols** heuristic intuitively.

## 🧩 Prerequisites
Notebooks 01–07 (the full PID controller and what each gain does).

## ⏱️ Estimated time
About **50–65 minutes**.

## 🧠 Concepts you know so far
Full PID · anti-windup · derivative filtering · the dashboard · rise/overshoot/settle/steady-state vocabulary

## 🔜 Concepts you'll learn here
Manual tuning workflow · per-gain sweeps · automatic metrics · Ziegler–Nichols heuristic


### 🔁 Quick recap — what each gain does (and its trade-off)

| Turn up… | You get more… | But risk… |
|---|---|---|
| **Kp** | speed / responsiveness | overshoot, oscillation |
| **Ki** | zero steady-state error | windup, slow oscillation |
| **Kd** | damping / smoothness | noise amplification, sluggishness |

Run setup and load our engine, controller, and a metrics helper.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


### 🧰 Engine + PID controller (from Notebook 07)

In [ ]:
from collections import deque

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=12.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone with ANY controller and return recorded signals as arrays.

    `controller` is a callable: controller(target, measured_altitude, dt) -> desired_thrust.
    It may hold internal state (integral, previous error, etc.) and may expose a dict
    `controller.last_terms = {'p':..,'i':..,'d':..}` which we record for the dashboards.

    Physical extras (all optional, default off):
      noise_std  : Gaussian sensor noise standard deviation (metres)
      wind       : constant extra force in newtons (+ up, - down)
      delay_steps: how many steps the sensor reading lags behind reality
      max/min_thrust : actuator saturation limits (newtons)
    """
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)  # sensor delay line

    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "p", "i", "d")
    hist = {k: [] for k in keys}

    for _ in range(int(total_time / dt)):
        buf.append(x)                                    # newest true altitude enters the line
        delayed = buf[0]                                 # controller sees the OLD reading
        measured = delayed + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = target - measured

        thrust = controller(target, measured, dt)        # <-- the controller decides
        thrust = min(max(thrust, min_thrust), max_thrust)  # actuator can't exceed limits

        net_force = thrust - mass * g + wind             # sum of forces (up +, down -)
        a = net_force / mass                             # Newton's second law

        terms = getattr(controller, "last_terms", {"p": 0.0, "i": 0.0, "d": 0.0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error,
                                 terms.get("p", 0.0), terms.get("i", 0.0), terms.get("d", 0.0))):
            hist[k].append(val)

        v = v + a * dt                                   # Euler integrate
        x = x + v * dt
        if x < 0:                                        # ground
            x, v = 0.0, 0.0
        t = t + dt

    return {k: np.array(val) for k, val in hist.items()}


In [ ]:
class PIDController:
    """A complete PID controller with anti-windup and a filtered derivative.

    control law:  thrust = Kp*error + Ki*(integral of error) + Kd*(derivative)
    - NO gravity feed-forward on purpose, so the INTEGRAL term is what learns to
      cancel gravity. This is why removing integral leaves a steady-state error.
    - derivative is taken on the MEASUREMENT (not the error) and low-pass filtered,
      which avoids 'derivative kick' and tames sensor noise.
    """
    def __init__(self, Kp=0.0, Ki=0.0, Kd=0.0,
                 i_limit=15.0, d_filter=0.1, use_anti_windup=True):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.i_limit = i_limit            # clamp on the integral term (anti-windup)
        self.d_filter = d_filter          # 1.0 = no smoothing, small = heavy smoothing
        self.use_anti_windup = use_anti_windup
        self.integral = 0.0
        self.prev_measured = None
        self.d_state = 0.0
        self.last_terms = {"p": 0.0, "i": 0.0, "d": 0.0}

    def __call__(self, target, measured, dt):
        error = target - measured

        # --- Proportional: react to the error RIGHT NOW ---
        p = self.Kp * error

        # --- Integral: accumulate error over time (with anti-windup clamp) ---
        self.integral += error * dt
        if self.use_anti_windup and self.Ki > 0:
            hi = self.i_limit / self.Ki
            self.integral = max(-hi, min(hi, self.integral))   # clamp integral
        i = self.Ki * self.integral

        # --- Derivative (on measurement, filtered): anticipate the future ---
        if self.prev_measured is None:
            raw = 0.0
        else:
            raw = (measured - self.prev_measured) / dt          # rate of change of altitude
        self.prev_measured = measured
        self.d_state += self.d_filter * (raw - self.d_state)    # low-pass filter
        d = -self.Kd * self.d_state                             # minus: opposes motion (damping)

        self.last_terms = {"p": p, "i": i, "d": d}
        return p + i + d


### 📏 A helper that measures a response automatically

In [ ]:
def performance_metrics(t, x, target, band=0.05):
    """Compute rise time, overshoot, settling time and steady-state error from a response."""
    t = np.asarray(t, float); x = np.asarray(x, float)
    final = float(np.mean(x[int(0.9 * len(x)):]))          # average of last 10% = steady state
    sse = target - final                                   # steady-state error (signed)

    thresh = 0.9 * target                                  # rise time = first reach 90% of target
    idx = np.where(x >= thresh)[0]
    rise = float(t[idx[0]]) if len(idx) else float("nan")

    peak = float(np.max(x))                                # overshoot = how far past target, in %
    overshoot = max(0.0, (peak - target) / target * 100.0)

    tol = band * target                                    # settling = last exit from +/-5% band
    outside = np.where(np.abs(x - target) > tol)[0]
    settle = float(t[outside[-1]]) if len(outside) else 0.0

    return dict(rise_time=rise, overshoot_pct=overshoot,
                settling_time=settle, steady_state_error=sse, final_value=final)


## 1. A repeatable manual tuning recipe

There are many recipes; here's a reliable beginner-friendly one. **Change one thing at a time** and
watch the response.

1. **Start** with `Ki = 0`, `Kd = 0`.
2. **Raise Kp** until the drone responds briskly and *just begins* to oscillate a little. Back off
   slightly. (Now it's fast but overshoots and sits a bit short.)
3. **Add Kd** to damp the overshoot and oscillation. Increase until the response is smooth. (Too
   much Kd → sluggish + noise-sensitive, so don't overdo it.)
4. **Add Ki** — a small amount — to erase the remaining steady-state error. Increase slowly until
   the drone reaches the target and stays. (Too much Ki → slow rolling oscillation.)
5. **Iterate** a little if needed. Good tuning is fast rise, small overshoot, quick settle, ~zero
   steady-state error.

Let's *watch* this recipe play out, stage by stage.


In [ ]:
stages = [
    ("1) Kp only (fast, overshoots, short)",      dict(Kp=4.0, Ki=0.0, Kd=0.0), "tab:orange"),
    ("2) + Kd (damped, still short)",             dict(Kp=4.0, Ki=0.0, Kd=4.0), "tab:green"),
    ("3) + Ki (exact + smooth = tuned!)",         dict(Kp=4.0, Ki=1.2, Kd=4.0), "tab:blue"),
]
plt.figure(figsize=(9, 5))
for name, gains, colour in stages:
    res = simulate(PIDController(**gains), total_time=16.0)
    plt.plot(res["t"], res["x"], color=colour, lw=2, label=name)
plt.axhline(10, color="black", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("The tuning recipe, stage by stage: P -> add D -> add I")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


## 2. Parameter sweeps — each gain's effect, in isolation

The cleanest way to build intuition: hold two gains fixed and **sweep** the third. Below are three
sweeps. Study the pattern in each.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

# Sweep Kp (Ki=0, Kd=0) -> speed vs overshoot
for Kp in [1, 2, 4, 8]:
    res = simulate(PIDController(Kp=Kp, Ki=0, Kd=0), total_time=12.0)
    axes[0].plot(res["t"], res["x"], lw=2, label=f"Kp={Kp}")
axes[0].set_title("Sweep Kp (higher = faster, more overshoot)"); axes[0].set_ylabel("altitude (m)")

# Sweep Kd (Kp=8 fixed) -> damping
for Kd in [0, 2, 5, 10]:
    res = simulate(PIDController(Kp=8, Ki=0, Kd=Kd), total_time=12.0)
    axes[1].plot(res["t"], res["x"], lw=2, label=f"Kd={Kd}")
axes[1].set_title("Sweep Kd at Kp=8 (higher = more damping)")

# Sweep Ki (Kp=4, Kd=4 fixed) -> removing steady-state error
for Ki in [0, 0.5, 1.5, 4.0]:
    res = simulate(PIDController(Kp=4, Ki=Ki, Kd=4), total_time=18.0)
    axes[2].plot(res["t"], res["x"], lw=2, label=f"Ki={Ki}")
axes[2].set_title("Sweep Ki at Kp=4,Kd=4 (higher = faster to exact, then oscillates)")

for ax in axes:
    ax.axhline(10, color="black", ls="--"); ax.set_xlabel("time (s)")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Kp: speed<->overshoot.  Kd: damping.  Ki: kills steady-state error (too much = slow oscillation).")


## 3. Make "good" measurable — automatic metrics

Eyeballing curves only gets you so far. Let's score several tunings with our metrics helper so you
can compare them numerically. Lower rise time, lower overshoot, lower settling time, and
near-zero steady-state error are what we want.


In [ ]:
candidates = {
    "under-tuned (Kp=1)":       dict(Kp=1.0, Ki=0.3, Kd=1.0),
    "over-tuned  (Kp=12,Kd=0)": dict(Kp=12.0, Ki=1.5, Kd=0.0),
    "well-tuned  (4,1.2,4)":    dict(Kp=4.0, Ki=1.2, Kd=4.0),
}
print(f"{'tuning':28} | {'rise(s)':>7} | {'overshoot%':>10} | {'settle(s)':>9} | {'sse(m)':>7}")
print("-"*76)
for name, gains in candidates.items():
    res = simulate(PIDController(**gains), total_time=20.0)
    m = performance_metrics(res["t"], res["x"], target=10.0)
    print(f"{name:28} | {m['rise_time']:>7.2f} | {m['overshoot_pct']:>10.1f} | "
          f"{m['settling_time']:>9.2f} | {m['steady_state_error']:>7.3f}")
print("\nThe well-tuned row wins across the board: quick rise, low overshoot, fast settle, ~0 error.")


## 4. Under-tuned vs over-tuned vs well-tuned, side by side


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (name, gains) in zip(axes, candidates.items()):
    res = simulate(PIDController(**gains), total_time=20.0)
    ax.plot(res["t"], res["x"], color="tab:blue", lw=2)
    ax.axhline(10, color="seagreen", ls="--")
    ax.set_title(name); ax.set_xlabel("time (s)"); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("altitude (m)")
plt.tight_layout(); plt.show()
print("Under-tuned: slow & lazy.  Over-tuned: violent overshoot/oscillation.  Well-tuned: just right.")


## 5. The Ziegler–Nichols heuristic (intuitive tour)

A famous starting-point recipe from 1942. The idea, in plain words:

1. Set `Ki = Kd = 0`. Slowly raise `Kp` until the system **oscillates with a steady, unchanging
   amplitude** — not growing, not shrinking. Call that gain the **ultimate gain `Ku`**, and the
   period of those oscillations `Tu`.
2. Plug `Ku` and `Tu` into simple formulas to get a first guess:

   `Kp = 0.6·Ku`,  `Ki = 1.2·Ku / Tu`,  `Kd = 0.075·Ku·Tu`  *(classic PID form)*

It's a **starting point**, not a final answer — you almost always fine-tune from there. Let's find
`Ku` for our drone by sweeping Kp until sustained oscillation appears.


In [ ]:
# Find roughly where sustained oscillation begins (our drone has no natural damping, so even
# modest Kp with no Kd rings for a long time -- we look for oscillation that barely decays).
plt.figure(figsize=(9, 5))
for Kp in [6, 10, 16, 24]:
    res = simulate(PIDController(Kp=Kp, Ki=0, Kd=0), total_time=14.0)
    plt.plot(res["t"], res["x"], lw=1.6, label=f"Kp={Kp}")
plt.axhline(10, color="black", ls="--")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Raising Kp toward the 'ultimate gain' Ku: oscillations grow and persist")
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 20); plt.show()

# Take Ku ~ 16 and read the oscillation period Tu off the curve (~1.5 s here) as a demo.
Ku, Tu = 16.0, 1.5
zn = dict(Kp=0.6*Ku, Ki=1.2*Ku/Tu, Kd=0.075*Ku*Tu)
print(f"Ziegler-Nichols first guess from Ku={Ku}, Tu={Tu}:")
print(f"   Kp = {zn['Kp']:.2f},  Ki = {zn['Ki']:.2f},  Kd = {zn['Kd']:.2f}")

res_zn = simulate(PIDController(**zn), total_time=18.0)
m = performance_metrics(res_zn["t"], res_zn["x"], 10.0)
print(f"Z-N response -> overshoot {m['overshoot_pct']:.0f}%, settle {m['settling_time']:.1f}s. "
      f"Classic Z-N is known for a fast but overshoot-heavy response -- a starting point to refine.")


## 6. 🎛️ Interactive tuner with live metrics

Tune by hand and get an instant scorecard. Try to beat the "well-tuned" numbers from Section 3:
low overshoot, fast settle, near-zero steady-state error.


In [ ]:
def tuner(Kp=4.0, Ki=1.2, Kd=4.0):
    res = simulate(PIDController(Kp, Ki, Kd), total_time=20.0)
    m = performance_metrics(res["t"], res["x"], 10.0)
    plt.figure(figsize=(9, 4.8))
    plt.plot(res["t"], res["x"], color="tab:blue", lw=2)
    plt.axhline(10, color="seagreen", ls="--")
    plt.axhspan(9.5, 10.5, color="seagreen", alpha=0.12)
    plt.ylim(0, 15); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
    plt.title(f"rise={m['rise_time']:.2f}s  overshoot={m['overshoot_pct']:.0f}%  "
              f"settle={m['settling_time']:.1f}s  sse={m['steady_state_error']:.3f}m")
    plt.grid(True, alpha=0.3); plt.show()

interact(tuner,
         Kp=FloatSlider(min=0.0, max=12.0, step=0.5, value=4.0),
         Ki=FloatSlider(min=0.0, max=5.0,  step=0.2, value=1.2),
         Kd=FloatSlider(min=0.0, max=12.0, step=0.5, value=4.0));


## ✅ Summary
- Tune with a **repeatable recipe**: raise **Kp** to briskness, add **Kd** to damp, add **Ki** to
  erase steady-state error — one gain at a time.
- **Parameter sweeps** reveal each gain's isolated effect; **metrics** make "good" measurable.
- **Ziegler–Nichols** gives a quick first guess from the ultimate gain `Ku` and period `Tu`, but
  it's a starting point you refine — it tends to overshoot.
- Good tuning = fast rise, small overshoot, quick settle, ~zero steady-state error.


## ⚠️ Common mistakes
- **Changing two gains at once.** You lose track of cause and effect. One at a time.
- **Chasing zero overshoot at all costs.** Often you trade away useful speed; a little overshoot is
  usually fine.
- **Treating Ziegler–Nichols output as final.** It's a first draft, famously overshoot-heavy.
- **Tuning on a noiseless, disturbance-free sim.** Gains that look perfect there can fail on real
  hardware (Notebook 09).

## 🧭 Mental model
> *"Kp for speed, Kd for smoothness, Ki for accuracy. Tune in that order, one knob at a time, and
> let the metrics — not just your eyes — tell you when to stop."*

## 🌍 Real-world applications
Every commissioned control loop gets tuned: drones, 3D printers, HVAC, motor drives, brewing and
chemical plants. Auto-tuners exist, but they use exactly these ideas under the hood.


## 🧪 Exercises

**L1 — Observe.** In the Kp sweep, what two things change as Kp increases from 1 to 8?
<details><summary>▸ Solution</summary>
It gets <b>faster</b> (shorter rise time) but shows <b>more overshoot</b> and ringing. Speed and
overshoot rise together — the core P trade-off.
</details>

**L2 — Modify.** Add your own tuning to the `candidates` dict (say `Kp=6, Ki=1.5, Kd=6`) and rerun
the metrics table. Can you get overshoot under 5% with settle under 6 s?
<details><summary>▸ Solution</summary>
Many combos work; a good region is roughly <code>Kp 3–6, Kd 3–6, Ki 1–2</code>. More Kd buys lower
overshoot; keep Ki modest so it doesn't slow-oscillate. Read the printed metrics to confirm.
</details>

**L3 — Predict.** Starting from the well-tuned set, you triple `Ki`. Predict what the metrics do,
then test.
<details><summary>▸ Solution</summary>
Steady-state error stays ~0, but you'll typically see **more overshoot and a slow rolling
oscillation** (the integral over-corrects), so settling time gets worse. Too much I hurts.
</details>


## ❓ Quick quiz
**Q1.** In the recommended recipe, which gain do you tune first?
<details><summary>▸ Answer</summary>**Kp** (with Ki=Kd=0), then add Kd, then Ki.</details>

**Q2.** Ziegler–Nichols needs which two measured quantities?
<details><summary>▸ Answer</summary>The **ultimate gain Ku** and the **oscillation period Tu**.</details>

**Q3.** Too much Ki tends to cause…
<details><summary>▸ Answer</summary>**Slow rolling oscillation** and extra overshoot (and worse settling).</details>

**Q4.** Why not trust a tuning that looks perfect on a clean sim?
<details><summary>▸ Answer</summary>Real systems have **noise, disturbances, delay, and saturation**; those can wreck an idealized tuning (Notebook 09).</details>


## 🔭 Preview of Notebook 09 — *Practical Control Issues*

Your tuned PID is about to meet the messy real world: **sensor noise, actuator saturation, wind
gusts, and measurement delay.** You'll see how each one degrades performance and how to defend
against it — building intuition for **robustness**, the art of performing well despite an imperfect
model and constant disturbances. 🚁
